# AgriLens AI — Crop Disease Detection Model Training Pipeline

Welcome to the complete AgriLens AI model training pipeline. This Colab/Jupyter Notebook shows you how to:
1. Download and set up the **PlantVillage / Kaggle Crop Disease Dataset**.
2. Run the automated data cleaning andMD5 deduplication pipelines.
3. Apply realistic crop-field data augmentations (zooms, rotations, noise, random shadows).
4. Train a lightweight **MobileNetV3Large** neural network backbone.
5. Evaluate accuracy, precision, recall, and generate Confusion Matrices.
6. Convert the model to **TensorFlow Lite** using Post-Training Quantization (Float16/Int8) for low-end Android smartphones.

## Step 1: Install Dependencies
Run this cell to set up the necessary Python libraries in your environment.

In [ ]:
!pip install tensorflow scikit-learn matplotlib pandas pillow opencv-python seaborn

## Step 2: Download PlantVillage Dataset
To download the dataset automatically using the Kaggle API, upload your `kaggle.json` credentials token and run the cells below.

*(Alternatively, download the PlantVillage dataset manually and place the unzipped images directory in `raw_dataset/`)*

In [ ]:
# If using Google Colab, upload kaggle.json here
from google.colab import files
uploaded = files.upload() # Select your kaggle.json token

In [ ]:
# Configure Kaggle token directory
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download plant disease dataset (~800MB)
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset
!unzip -q new-plant-diseases-dataset.zip -d raw_dataset

## Step 3: Run Data Cleaning and Split
We instantiate our `DatasetProcessor` to filter duplicates, exclude corrupted images, and split valid files into `train` (70%), `val` (20%), and `test` (10%) splits.

In [ ]:
from dataset_processor import DatasetProcessor

# Point this path to your unzipped category folders
raw_dir = "raw_dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
processor = DatasetProcessor(raw_data_dir=raw_dir, output_data_dir="data")
stats = processor.clean_and_split()

## Step 4: Load & Augment Datasets
We load datasets with TensorFlow, normalize pixels from `[0, 255]` to `[0, 1]`, and map our custom augmentation pipelines (which simulate field shadows and brightness levels).

In [ ]:
from train import CropDiseaseTrainer

trainer = CropDiseaseTrainer(data_dir="data", epochs=20)
trainer.load_datasets()

## Step 5: Build and Train MobileNetV3Large
Compile the MobileNetV3Large neural network, configure callbacks (Early Stopping, Learning Rate Plateau), and initiate the training run.

In [ ]:
trainer.run_training(model_type="mobilenet")

## Step 6: Evaluate Model & Confusion Matrix
Run evaluations on the test split to compute precision, recall, and plot the confusion matrix.

In [ ]:
trainer.evaluate_model()

## Step 7: Convert Model to TensorFlow Lite
We convert the Keras `.h5` model to `.tflite` using Post-Training Quantization (PTQ) to ensure low RAM consumption and fast CPU processing on budget Android phones.

In [ ]:
from tflite_converter import TFLiteModelConverter, representative_data_generator

converter = TFLiteModelConverter(keras_model_path="best_crop_model.h5")

# 1. Convert to Float16 (optimized size + GPU execution)
converter.convert_to_float16(output_path="agri_model_float16.tflite")

# 2. Convert to Int8 (optimized for low-end mobile CPU and low RAM)
converter.convert_to_int8(
    representative_data_gen=representative_data_generator,
    output_path="agri_model.tflite"
)

## Step 8: Verify Local Inference (Python)
Verify the converted `.tflite` model works by running a sample image through the TensorFlow Lite interpreter.

In [ ]:
import numpy as np
import tensorflow as tf

# Load TFLite model and allocate tensors
interpreter = tf.lite.Interpreter(model_path="agri_model.tflite")
interpreter.allocate_tensors()

# Get input and output tensors details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input Details:", input_details)
print("Output Details:", output_details)

# Generate mock test input
mock_input = np.random.rand(1, 224, 224, 3).astype(np.float32)
if input_details[0]['dtype'] == np.uint8:
    mock_input = (mock_input * 255).astype(np.uint8)

interpreter.set_tensor(input_details[0]['index'], mock_input)
interpreter.invoke()

output_data = interpreter.get_tensor(output_details[0]['index'])
print("Model raw inference predictions:", output_data)
print("Predicted class index:", np.argmax(output_data))